<a href="https://colab.research.google.com/github/Troptrap/Animatediff-colab/blob/main/Animatediff.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# 1. INSTALL COMFYUI & REQUIRED CUSTOM NODES
# ==========================================
!apt-get install -y aria2
!git clone https://github.com/comfyanonymous/ComfyUI
%cd /content/ComfyUI

# Install core python requirements
!pip install -r requirements.txt
!pip install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121

# Install AnimateDiff Evolved, VideoHelperSuite, IP-Adapter Plus, and ComfyUI-Manager
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/Kosinkadink/ComfyUI-AnimateDiff-Evolved
!git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite
!git clone https://github.com/cubiq/ComfyUI_IPAdapter_plus
!git clone https://github.com/ltdrdata/ComfyUI-Manager

# Install custom node dependencies
%cd /content/ComfyUI/custom_nodes/ComfyUI-VideoHelperSuite
!pip install -r requirements.txt
!pip install -U --pre comfyui-manager

# ==========================================
# 2. DOWNLOAD MODELS, MOTION MODULES & LORAS
# ==========================================

# A. Base Checkpoint: DreamShaper 8 (SD 1.5)
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
  "https://huggingface.co/Lykon/DreamShaper/resolve/main/DreamShaper_8_pruned.safetensors" \
  -d /content/ComfyUI/models/checkpoints -o DreamShaper_8_pruned.safetensors

# B. AnimateDiff v2 Motion Module
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
  "https://huggingface.co/guoyww/animatediff/resolve/main/mm_sd_v15_v2.ckpt" \
  -d /content/ComfyUI/custom_nodes/ComfyUI-AnimateDiff-Evolved/models -o mm_sd_v15_v2.ckpt

# C. AnimateLCM Motion Module (Fast 4-8 Step Video Generation)
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
  "https://huggingface.co/wangfuyun/AnimateLCM/resolve/main/AnimateLCM_sd15_t2v.ckpt" \
  -d /content/ComfyUI/custom_nodes/ComfyUI-AnimateDiff-Evolved/models -o AnimateLCM_sd15_t2v.ckpt

# D. AnimateLCM LoRA
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
  "https://huggingface.co/wangfuyun/AnimateLCM/resolve/main/AnimateLCM_sd15_t2v_lora.safetensors" \
  -d /content/ComfyUI/models/loras -o AnimateLCM_sd15_t2v_lora.safetensors

# E. Standard SD 1.5 VAE
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
  "https://huggingface.co/stabilityai/sd-vae-ft-mse-original/resolve/main/vae-ft-mse-840000-ema-pruned.safetensors" \
  -d /content/ComfyUI/models/vae -o vae-ft-mse-840000-ema-pruned.safetensors

# F. CLIP Vision Encoder (Required for IP-Adapter)
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
  "https://huggingface.co/h94/IP-Adapter/resolve/main/models/image_encoder/model.safetensors" \
  -d /content/ComfyUI/models/clip_vision -o CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors

# G. IP-Adapter Plus Weights (Image-to-Video Driver)
!mkdir -p /content/ComfyUI/models/ipadapter
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
  "https://huggingface.co/h94/IP-Adapter/resolve/main/models/ip-adapter-plus_sd15.safetensors" \
  -d /content/ComfyUI/models/ipadapter -o ip-adapter-plus_sd15.safetensors

# ==========================================
# 3. LAUNCH COMFYUI WITH LOCALTUNNEL (FIXED)
# ==========================================
%cd /content/ComfyUI

import subprocess
import time
import urllib.request

# Start ComfyUI in the background
subprocess.Popen(["python", "main.py", "--dont-print-server", "--enable-manager", "--highvram", "--listen", "127.0.0.1", "--port", "8188"])

# Get your Colab IP password required by localtunnel
ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()

print("\n" + "="*50)
print(f"🔑 LOCALTUNNEL PASSWORD: {ip}")
print("(Copy this password, you will enter it at the link below)")
print("="*50 + "\n")

# Install and launch localtunnel
!npm install -g localtunnel > /dev/null
!npx localtunnel --port 8188

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
aria2 is already the newest version (1.36.0-1).
0 upgraded, 0 newly installed, 0 to remove and 57 not upgraded.
fatal: destination path 'ComfyUI' already exists and is not an empty directory.
/content/ComfyUI
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121
/content/ComfyUI/custom_nodes
fatal: destination path 'ComfyUI-AnimateDiff-Evolved' already exists and is not an empty directory.
fatal: destination path 'ComfyUI-VideoHelperSuite' already exists and is not an empty directory.
fatal: destination path 'ComfyUI_IPAdapter_plus' already exists and is not an empty directory.
fatal: destination path 'ComfyUI-Manager' already exists and is not an empty directory.
/content/ComfyUI/custom_nodes/ComfyUI-VideoHelperSuite
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.0 MB/s eta 0:00:00

Download Results:
gid   |stat|avg speed  |path/URI
======+====+====